<!-- 학습 보강 셀 -->

# 11. ChromaDB 학습 흐름

이 노트북은 ChromaDB를 영구 저장 가능한 벡터 DB로 사용하는 예제입니다.
FAISS가 로컬 벡터 검색 라이브러리에 가깝다면, ChromaDB는 컬렉션 단위로 데이터를 관리하는 벡터 데이터베이스에 가깝습니다.

In [11]:
# ChromaDB 벡터 스토어 예제에 필요한 패키지 설치
# - 패키지명은 llama-index-vector-stores-chroma 입니다. 기존의 점(.) 표기는 pip 패키지명으로 잘못된 형식입니다.
# !pip install chromadb llama-index-vector-stores-chroma llama-index-llms-ollama llama-index-embeddings-ollama

In [12]:
import gc
import shutil
from pathlib import Path

import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [13]:
# OLLAMA_MODEL_PREP_CELL
# Ollama 모델은 pip/requirements.txt로 설치되지 않습니다.
# 이 셀은 노트북 실행 전에 필요한 로컬 Ollama 모델이 있는지 확인하고, 없으면 자동으로 pull 합니다.
import subprocess

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_LLM_MODEL = 'gemma2:2b'
OLLAMA_EMBED_MODEL = 'nomic-embed-text'

def _installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = _installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL]:
    ensure_ollama_model(model_name)

이미 설치됨: gemma2:2b
이미 설치됨: nomic-embed-text


In [14]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    temperature=0.5,
    request_timeout=120,
    base_url=OLLAMA_BASE_URL,
)

embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

In [15]:
# 데이터 로드
documents = SimpleDirectoryReader('../NewData/pdf_sample2/').load_data()
print('읽어온 문서 수:', len(documents))

읽어온 문서 수: 11


In [16]:
# ChromaDB 영구 저장 클라이언트 생성
# - path에 지정한 디렉토리에 벡터 DB가 저장됩니다.
# - 커널이 열린 상태에서 chroma_db 폴더만 삭제하면 이전 SQLite 연결이 남아 readonly 오류가 날 수 있습니다.
# - 깨끗하게 다시 만들 때는 RESET_CHROMA_DB=True로 두고 이 셀부터 다시 실행하세요.
CHROMA_DB_PATH = Path('./chroma_db').resolve()
RESET_CHROMA_DB = True

if RESET_CHROMA_DB:
    for name in ['index', 'query_engine', 'vector_store', 'storage_context', 'chroma_collection', 'db']:
        globals().pop(name, None)
    gc.collect()

    # Chroma는 같은 프로세스 안에서 PersistentClient 시스템을 캐시하므로 재생성 전에 비웁니다.
    from chromadb.api.client import SharedSystemClient
    SharedSystemClient.clear_system_cache()

    if CHROMA_DB_PATH.exists():
        shutil.rmtree(CHROMA_DB_PATH)

db = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

# 컬렉션 생성 또는 로드
# - 기존 실습과 동일하게 quickstart_ollama 컬렉션을 사용합니다.
chroma_collection = db.get_or_create_collection('quickstart_ollama')

<!-- 학습 보강 셀 -->

## 컬렉션을 사용하는 이유

ChromaDB의 컬렉션은 관련 벡터들을 묶는 단위입니다.
프로젝트별, 문서 종류별, 임베딩 모델별로 컬렉션을 나누면 데이터를 관리하고 비교하기 쉬워집니다.

In [17]:
# ChromaDB를 LlamaIndex의 인덱싱 및 검색 파이프라인에 통합합니다.
vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

<!-- 학습 보강 셀 -->

## 반복 실행 시 중복 데이터 주의

`get_or_create_collection`은 기존 컬렉션이 있으면 그대로 재사용합니다.
같은 노트북을 여러 번 실행하면 같은 문서가 중복으로 들어갈 수 있으므로, 실험을 처음부터 다시 하려면 컬렉션이나 `chroma_db` 디렉토리를 정리해야 합니다.

In [18]:
# 인덱스 생성 및 데이터 임베딩
# - 같은 컬렉션에 반복 실행하면 중복 문서가 쌓일 수 있습니다.
# - 깨끗하게 다시 만들고 싶다면 chroma_db 디렉토리나 컬렉션을 삭제한 뒤 실행하세요.
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

Generating embeddings: 100%|██████████| 12/12 [00:01<00:00,  7.58it/s]


<!-- 학습 보강 셀 -->

## ChromaDB 저장 후 다음 노트북과 연결되는 지점

이 셀에서 생성된 벡터는 `./chroma_db`에 저장됩니다.
따라서 12번 노트북은 원본 PDF를 다시 읽지 않고, 이 저장된 ChromaDB 컬렉션을 바로 열어 질의합니다.

---
### 메모리 인덱스 질의

In [19]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

2026-06-02 11:38:14,238 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [20]:
# 쿼리 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)

2026-06-02 11:38:14,267 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:38:17,563 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문: 이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘
답변: 이 논문에서는 Transformer 모델이 다른 모델들보다 우수한 성능을 보여주며, 특히 효율적인 학습과 높은 BLEU 점수를 보였습니다.  

